# [2장 2강] - 경사하강법, 옵티마이저 및 데이터 분할
---
## 실습 목표
- 데이터 분할 규칙에 따라 데이터를 훈련·검증·테스트 세트로 분류할 수 있다.
- 표준화 스케일러를 활용하여 데이터 단위를 일정하게 조정할 수 있다.
- 경사하강법 하이퍼파라미터를 직접 설정하고 분할된 데이터를 검증할 수 있다.

### 필수 1 : 범죄율 데이터 필터링 및 훈련, 검증, 테스트 데이터 세트 분할
#### 문제 1-1 :  가족형 주택 기준 데이터 분할하기

요구 사항
---
1. pandas 라이브러리를 사용하여 usa_housing.csv 파일을 불러오세요.
2. Bedrooms 변수의 값이 3 이상인 데이터만 필터링하여 새로운 데이터프레임으로 저장하세요.
3. 입력 데이터인 CrimeRate 변수와 타깃(정답) 데이터인 Price 변수를 각각 2차원 배열 형태로 추출하세요.
4. train_test_split 함수를 사용하여 전체 데이터를 훈련 세트 60%, 검증 세트 20%, 테스트 세트 20%의 비율로 분할하세요. 이때 random_state 값은 42로 고정하세요.
5. 분할이 완료된 후 각 데이터 세트의 샘플 개수를 print() 함수로 출력하여 정상적으로 분할되었는지 확인하세요.

**출력 및 검증 방법**
훈련 세트, 검증 세, 테스트 세트의 데이터 개수가 각각 명확한 숫자로 출력되는지 확인하세요. 각 데이터 개수가 출력된 부분을 캡처하여 작성한 코드와 함께 제출하세요.

In [23]:
import pandas as pd
df = pd.read_csv("usa_housing.csv")
df = df[df["Bedrooms"]>=3]

df.head()

,Price,Bedrooms,Bathrooms,SquareFeet,YearBuilt,GarageSpaces,LotSize,ZipCode,CrimeRate,SchoolRating
3,465838,3,3.3,2708,1907,3,1.62,80587,61.65,1
4,359178,4,3.4,1175,1994,2,0.74,20756,15.66,4
9,737147,3,2.6,4191,2017,0,1.65,66549,37.60,3
10,621430,4,2.3,3765,1979,2,0.96,44503,55.01,9
13,275203,4,1.4,1672,1924,1,1.49,84069,27.38,2


In [24]:
X = df[["CrimeRate"]]
y = df["Price"]

In [25]:
from sklearn.model_selection import train_test_split

X_train_tmp, X_test, y_train_tmp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_tmp,y_train_tmp,
    test_size=0.25,
    random_state=42
)

print("훈련 세트 개수:", len(X_train))
print("검증 세트 개수:", len(X_valid))
print("테스트 세트 개수:", len(X_test))

훈련 세트 개수: 104
검증 세트 개수: 35
테스트 세트 개수: 35


### 필수 2 : StandardScaler를 활용한 데이터 표준화
#### 문제 2-1 :  데이터 표준화 스케일링 적용하기

**요구사항**

---

1. scikit-learn 라이브러리에서 제공하는 StandardScaler를 불러오세요.
2. 입력 데이터용 스케일러와 타깃 데이터용 스케일러를 각각 독립적으로 생성하세요.
3. fit_transform() 함수를 사용하여 앞서 분할한 데이터들을 표준 점수로 변환하세요.
4. 변환이 끝난 후, 후속 연산의 편의를 위해 flatten() 함수를 사용하여 데이터를 1차원 배열 형태로 변환하세요.
5. 변환된 데이터의 첫 번째 값과 전체 크기(shape)를 print() 함수를 통해 화면에 출력하세요.

**출력 및 검증 방법**
출력창에 표준화된 첫 번째 데이터 값과 전체 데이터의 크기가 정상적으로 표시되는지 확인하세요. 해당 출력 결과가 담긴 화면을 캡처하여 작성된 코드와 함께 제출하세요.

In [26]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()

X_scaled = ss.fit_transform(X).flatten()


print(X_scaled[0])
X_scaled.shape

0.39643667256090737


(174,)

### 심화 1 : 학습률 조정을 위한 하이퍼파라미터 초기화 및 경사하강법 준비
#### 문제 3-1 : 학습률 설정 및 오차 기록 리스트 생성하기

**요구사항**

---

1. 모델의 파라미터인 가중치와 편향의 초기값을 모두 0.0으로 설정하세요.
2. 하이퍼파라미터인 학습률을 0.05로 설정하세요.
3. 총 학습 횟수인 에포크(Epoch)를 150회로 지정하세요.
4. 훈련 과정과 검증 과정에서 발생하는 손실(Loss)의 변화를 기록할 수 있도록 빈 리스트 train_losses와 valid_losses를 각각 생성하세요.
5. 설정된 초기 가중치, 학습률, 그리고 생성된 리스트 변수들을 print() 함수로 출력하여 변수가 정상적으로 정의되었는지 확인하세요.

**출력 및 검증 방법**
초기 가중치 값 0.0과 설정한 학습률 0.05, 그리고 리스트의 값들이 에러 없이 정상적으로 출력되는지 확인하세요. 출력 전체 화면을 캡처하여 작성된 코드와 함께 제출하세요.

In [27]:
import numpy as np

weight = 0.0          # 가중치(w): 집 크기에 따른 가격 영향력 (초기값 0)
bias = 0.0            # 편향(b): 기본적으로 깔고 가는 베이스 집값 (초기값 0)
learning_rate = 0.05   # 보폭(학습률): 스케일링을 했으므로 보폭을 조금 넓게 잡아도 안전합니다!
epochs = 150         # 훈련 횟수: 산을 내려가는 걸음을 총 200번 반복합니다.

# 훈련 과정에서 오차가 어떻게 줄어드는지 기록할 빈 리스트(수첩)를 만듭니다.
train_losses = []
valid_losses = []
history_w = []

N_train = len(X_train)
N_valid = len(X_valid)

In [28]:
# 경사하강법 루프 시작 (200번 반복)
for epoch in range(epochs):
    # 훈련 세트: 현재 가중치로 집값을 예측해 봅니다 (y_pred = w * x + b)
    y_pred_train = weight * X_train + bias
    
    # 예측 집값과 실제 집값 차이의 제곱 평균(MSE Loss)을 구합니다.
    train_loss = np.mean((y_pred_train - y_train) ** 2)
    train_losses.append(train_loss)
    
    # 검증 세트: 모의고사 데이터를 넣어 현재 실력을 중간 점검합니다.
    y_pred_valid = weight * X_valid + bias
    valid_loss = np.mean((y_pred_valid - y_valid) ** 2)
    valid_losses.append(valid_loss)
    history_w.append(weight)
    
    # 기울기(Gradient) 구하기: 어느 방향으로 가야 오차가 줄어들지 미분 공식으로 찾습니다.
    dw = (2 / N_train) * np.sum((y_pred_train - y_train) * X_train)
    db = (2 / N_train) * np.sum(y_pred_train - y_train)
    
    # 한 걸음 이동! (현재 위치 - 보폭 * 기울기)
    weight -= learning_rate * dw
    bias -= learning_rate * db

In [29]:
history_w[:10]

[0.0,
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0)]